# 02 · Notebook chung cho mọi run (Bước 4–6, FR-TRN-01) — Runtime: T4 GPU
**Chỉ sửa ô 5 (`RUN_ID`).** Mất phiên thì chạy lại từ ô 1 với cùng `RUN_ID`: run tự resume từ `last.pt` trên Drive.

Thứ tự run: `pilot` (ô 6b) → `base` → `random_s0`, `meta_s0`, `tool_s0` → `*_s1` → `*_s2`.
Sau mỗi lần đổi trạng thái, cập nhật `results/runs_registry.csv` trên Mac.

In [ ]:
# Ô 1 — Drive, GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
# Ô 2 — code (lần đầu clone, các lần sau pull) và thư viện
import os
REPO_URL = "https://github.com/<user>/build-phase-individual-n3-07b.git"   # <-- sửa <user>
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements-colab.txt

In [ ]:
# Ô 3 — dữ liệu: chép zip từ Drive sang ổ Colab rồi giải nén (bỏ qua nếu đã có)
RAW = "/content/drive/MyDrive/n3-07b/raw"
if not os.path.exists("/content/data/bdd100k/images/100k/val"):
    !mkdir -p /content/data
    for f in ["100k_images_train.zip", "100k_images_val.zip"]:
        !cp {RAW}/{f} /content/ && unzip -q -n /content/{f} -d /content/data && rm /content/{f}
    !unzip -q -n {RAW}/yolo_labels.zip -d /content/data/bdd100k
!sha256sum {RAW}/yolo_labels.zip   # so với docs/bdd100k.sha256 (FR-TRN-07)
!ls /content/data/bdd100k/labels/100k/train | wc -l

In [ ]:
# Ô 4 — đường dẫn
os.environ["DATA_ROOT"] = "/content/data"
os.environ["RUNS_ROOT"] = "/content/drive/MyDrive/n3-07b/runs"

In [ ]:
# Ô 5 — ô DUY NHẤT được sửa
RUN_ID = "base"

In [ ]:
# Ô 6 — kiểm split rồi train (tự resume)
!python -m src.data.check_splits && python -m src.train.train_run --run-id {RUN_ID}

In [ ]:
# Ô 6b — CHỈ dùng cho pilot (Bước 4.3): 3 epoch trên 20% base.
# Khi epoch 2 đang chạy: Runtime → Interrupt execution, rồi chạy lại ô này → log phải báo resume.
# !python -m src.train.train_run --run-id pilot --epochs 3 --fraction 0.2

In [ ]:
# Ô 7 — chấm điểm sau khi train xong (last.pt, rare slice + val2k)
!python -m src.train.eval_detector --run-id {RUN_ID}

In [ ]:
# Ô 8 — CHỈ chạy một lần sau khi base xong (Bước 5.2): uncertainty trên 40.000 ảnh pool
# !python -m src.uncertainty.run_pool